# 📁 Notebook 01: Data Ingestion
## Chest Cancer Classification using CNN and MLOps

**Dataset:** [Chest CT-Scan Images](https://www.kaggle.com/datasets/mohamedhanyyy/chest-ctscan-images)

**Classes:**
- Adenocarcinoma
- Large Cell Carcinoma
- Squamous Cell Carcinoma
- Normal

**Goals of this notebook:**
1. Load and explore the dataset structure
2. Count images per class and split
3. Visualize sample images
4. Identify class imbalance


In [ ]:
# ─── Standard Imports ────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# ─── Add src to path ─────────────────────────────────────────────
sys.path.append('..')
from src.data_ingestion import count_images, validate_structure, print_stats

# ─── Style ───────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All imports successful!')

In [ ]:
# ─── Configuration ───────────────────────────────────────────────
RAW_DIR   = Path('../data/raw')
SPLITS    = ['train', 'test', 'valid']
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

print(f'Dataset root: {RAW_DIR.resolve()}')
print(f'Exists: {RAW_DIR.exists()}')

## 📊 Step 1: Validate Dataset Structure

In [ ]:
# ─── Validate folder structure ───────────────────────────────────
def validate_and_list(raw_dir: Path):
    """Walk directory tree and list all splits and classes."""
    if not raw_dir.exists():
        print(f'⚠️  Dataset not found at: {raw_dir}')
        print('Please download from Kaggle:')
        print('  kaggle datasets download -d mohamedhanyyy/chest-ctscan-images')
        print('  unzip chest-ctscan-images.zip -d data/raw/')
        return
    
    print('📂 Dataset Structure:')
    print('=' * 50)
    for split in SPLITS:
        split_path = raw_dir / split
        if split_path.exists():
            classes = [d.name for d in split_path.iterdir() if d.is_dir()]
            print(f'  {split}/')
            for cls in sorted(classes):
                print(f'    ├── {cls}/')
        else:
            print(f'  {split}/ ← NOT FOUND')

validate_and_list(RAW_DIR)

## 📊 Step 2: Count Images Per Class

In [ ]:
# ─── Count images ────────────────────────────────────────────────
def count_images_nb(raw_dir: Path) -> dict:
    """Count images in each split/class folder."""
    counts = {}
    total  = defaultdict(int)
    
    for split in SPLITS:
        split_path = raw_dir / split
        if not split_path.exists():
            counts[split] = {}
            continue
        
        split_counts = {}
        for class_dir in sorted(split_path.iterdir()):
            if class_dir.is_dir():
                imgs = [f for f in class_dir.iterdir() 
                        if f.suffix.lower() in IMAGE_EXT]
                split_counts[class_dir.name] = len(imgs)
                total[class_dir.name] += len(imgs)
        
        counts[split] = split_counts
    
    counts['total'] = dict(total)
    return counts

counts = count_images_nb(RAW_DIR)

# Print as DataFrame
df_counts = pd.DataFrame({
    split: counts.get(split, {}) 
    for split in SPLITS
}).fillna(0).astype(int)

df_counts['Total'] = df_counts.sum(axis=1)
print('\n📊 Image Count per Class per Split:')
print('=' * 50)
print(df_counts.to_string())
print('=' * 50)
print(f'Grand Total: {df_counts["Total"].sum()} images')

## 📊 Step 3: Visualize Class Distribution

In [ ]:
# ─── Bar chart: images per class ─────────────────────────────────
if df_counts.empty or df_counts['Total'].sum() == 0:
    # Demo with dummy data when dataset not present
    demo_data = {
        'Class': ['Adenocarcinoma', 'Large Cell Carcinoma', 'Normal', 'Squamous Cell Carcinoma'],
        'Train': [70, 70, 70, 70],
        'Valid': [20, 20, 20, 20],
        'Test':  [33, 33, 33, 33],
    }
    df_plot = pd.DataFrame(demo_data).set_index('Class')
    df_plot['Total'] = df_plot.sum(axis=1)
    print('⚠️  Using demo data — place real dataset at data/raw/')
else:
    df_plot = df_counts

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Per split bar chart ──────────────────────────────────────────
splits_to_plot = [c for c in df_plot.columns if c != 'Total']
df_plot[splits_to_plot].plot(
    kind='bar', ax=axes[0], colormap='tab10', edgecolor='black', linewidth=0.5
)
axes[0].set_title('Images per Class per Split', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cancer Type', fontsize=11)
axes[0].set_ylabel('Number of Images', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Split')

# ── Pie chart: total distribution ───────────────────────────────
axes[1].pie(
    df_plot['Total'],
    labels=df_plot.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('husl', len(df_plot)),
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Overall Class Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Chest CT-Scan Dataset Overview', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

## 🖼️ Step 4: Display Sample Images

In [ ]:
# ─── Sample image grid ───────────────────────────────────────────
def show_sample_images(raw_dir: Path, split: str = 'train', n_per_class: int = 3):
    """Display n sample images for each class in a grid."""
    split_path = raw_dir / split
    
    if not split_path.exists():
        print(f'⚠️  No data at {split_path}. Download dataset first.')
        return
    
    class_dirs = sorted([d for d in split_path.iterdir() if d.is_dir()])
    n_classes  = len(class_dirs)
    
    if n_classes == 0:
        print('No class folders found.')
        return
    
    fig, axes = plt.subplots(n_classes, n_per_class, 
                              figsize=(n_per_class * 4, n_classes * 3.5))
    
    if n_classes == 1:
        axes = [axes]
    
    for row, class_dir in enumerate(class_dirs):
        images = sorted([
            f for f in class_dir.iterdir()
            if f.suffix.lower() in IMAGE_EXT
        ])[:n_per_class]
        
        for col in range(n_per_class):
            ax = axes[row][col] if n_per_class > 1 else axes[row]
            
            if col < len(images):
                img = mpimg.imread(str(images[col]))
                ax.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
                if col == 0:
                    ax.set_ylabel(class_dir.name.replace('.', '\n'), 
                                  fontsize=10, fontweight='bold')
                ax.set_title(f'Sample {col+1}', fontsize=9)
            else:
                ax.axis('off')
            
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle(f'Sample CT-Scan Images ({split.upper()} set)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../data/processed/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_sample_images(RAW_DIR, split='train', n_per_class=3)

## 💾 Step 5: Save Class Counts for DVC

In [ ]:
# ─── Save counts JSON ────────────────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/class_counts.json', 'w') as f:
    json.dump(counts, f, indent=2)

print('✅ class_counts.json saved!')
print(json.dumps(counts, indent=2))

## ✅ Summary

| Item | Detail |
|------|--------|
| Dataset | Chest CT-Scan Images (Kaggle) |
| Classes | 4 (Adenocarcinoma, Large Cell, Normal, Squamous) |
| Splits  | Train / Validation / Test |
| Next Step | Preprocessing & Augmentation |

➡️ **Next:** Open `02_preprocessing.ipynb`